# Day 1 - Sprint 3 Planning & NLP Preprocessing

## Google Play Store Review Sentiment Classification

**Dataset:** Google Play Store User Reviews  
**NLP Task:** Sentiment Classification  
**Main text column:** `Translated_Review`  
**Target column:** `Sentiment`  
**Previous core model:** DistilBERT Transformer

---

## Learning Objectives

Upon completing this notebook, we will be able to:

- Perform Sprint 3 planning and establish integration & evaluation backlog.
- Move on to Sprint 3 after taking lessons learned from Sprint 2 retrospective.
- Understand the need for preprocessing of raw text.
- Use word tokenization.
- Understand sub-word tokenization technique used by Transformers.
- Use lowercasing and punctuation cleaning.
- Perform stop-word removal while preserving sentiment-relevant negations.
- Apply lemmatization.
- Compare lemmatization to stemming.
- Build NLP preprocessing pipeline.
- Apply NLP preprocessing pipeline to our Google Play reviews dataset.
- Compare original text and cleaned one.
- Make sure task-relevant words like `not`, `no`, and `never` are kept.
- Describe preprocessing decisions made and their limitations.

---

## Notebook Content

1. Sprint 3 Planning
2. Environment Setup
3. Load Google Play Reviews Dataset
4. Understand Text Data
5. Why is Preprocessing Needed?
6. Word Tokenization
7. Sub-word Tokenization
8. Text Cleaning and Normalization
9. Stop-word Removal and Negation Preservation
10. Lemmatization
11. Lemmatization vs. Stemming
12. Full NLP Preprocessing Pipeline
13. Apply Preprocessing to the Project Dataset
14. Before vs. After Comparison
15. Verify Task-Critical Words
16. Document Preprocessing Decisions
17. Day 1 Requirements Checklist
18. Final Summary

#  Sprint 3 Planning

## Sprint Goal

The goal of Sprint 3 is to integrate the NLP preprocessing and trained model into a complete pipeline and evaluate the final system rigorously before deployment in Sprint 4.
The sprint moves the project from an individually trained model toward a complete and trustworthy component that can be prepared for deployment.

## Sprint 3 Backlog

| Task            | Description                             | Priority     |
|-----------------|----------------------------------------|--------------|
| NLP Preprocessing | Cleaning and normalization of review text | High         |
| Tokenization    | Tokenizing text and understanding Transformer's tokenization | High         |
| Stop-word       | Removing words that have low signal value without losing information about sentiments | High         |
| Lemmatization   | Transforming words to base form        | High         |
| Pipeline        | Integrating preprocessing with model pipeline | High         |
| Model           | Evaluating the model with Accuracy, Precision, Recall and F1-score metrics | High         |
| Error analysis  | Reviewing wrong predictions and hard cases  | Medium       |
| Documentation   | Logging decision regarding preprocessing and evaluation | Medium       |

### Day 1 Tasks

Today we will mainly finish:

- Planning Sprint 3.
- NLP preprocessing.
- Text cleaning.
- Tokenization.
- Stop-word handling.
- Lemmatization.
- Verifying words that are critical for the task.
- Logging preprocessing decisions.




## Improvements Implemented from Sprint 2

From the earlier analysis for model development, the following improvements will be implemented in Sprint 3:

- Stick with **DistilBERT** as the NLP architecture as it outperformed the text LSTM model in the previous comparison.
- Employ several evaluation metrics instead of depending on accuracy alone.
- Do not discard words that have the ability to alter sentiment, especially negations.
- Preprocess with caution rather than deleting information blindly.
- Aim for completeness in the pipeline prior to deployment.

### Project Background Information

In the previous project, the same Google Play review dataset was used but with a comparison between a Text LSTM and DistilBERT Transformer model. The sentiment classes in the dataset include:

- Positive
- Neutral
- Negative

The relevant columns are `Translated_Review` and `Sentiment`.

#  Environment Setup

We will use **Google Colab** with Python, Pandas and NLTK.

The notebook downloads the same Google Play Store user-review dataset used in Day 4.

In [33]:
!pip -q install kagglehub nltk scikit-learn transformers

In [34]:
import os
import re
import string
import numpy as np
import pandas as pd
import nltk

from collections import Counter
from IPython.display import display

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

print("Environment setup completed.")

Environment setup completed.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


#  Load the Google Play Reviews Dataset

This is the same dataset used in Day 4.

The relevant file is:

`googleplaystore_user_reviews.csv`

The columns we need are:

- `Translated_Review` → input text
- `Sentiment` → target label

We will download the dataset automatically so the notebook can run directly in Colab.

In [35]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "lava18/google-play-store-apps"
)

print("Dataset downloaded to:")
print(dataset_path)

print("\nFiles found:")
for root, dirs, files in os.walk(dataset_path):
    for file in files:
        print(os.path.join(root, file))

Using Colab cache for faster access to the 'google-play-store-apps' dataset.
Dataset downloaded to:
/kaggle/input/google-play-store-apps

Files found:
/kaggle/input/google-play-store-apps/googleplaystore.csv
/kaggle/input/google-play-store-apps/license.txt
/kaggle/input/google-play-store-apps/googleplaystore_user_reviews.csv


In [36]:
reviews_path = None

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.lower() == "googleplaystore_user_reviews.csv":
            reviews_path = os.path.join(root, file)
            break
    if reviews_path is not None:
        break

print("Reviews file:")
print(reviews_path)

if reviews_path is None:
    raise FileNotFoundError(
        "googleplaystore_user_reviews.csv was not found."
    )

Reviews file:
/kaggle/input/google-play-store-apps/googleplaystore_user_reviews.csv


In [37]:
reviews_df = pd.read_csv(reviews_path)

print("Dataset shape:", reviews_df.shape)
display(reviews_df.head())

print("\nColumns:")
print(reviews_df.columns.tolist())

Dataset shape: (64295, 5)


,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000



Columns:
['App', 'Translated_Review', 'Sentiment', 'Sentiment_Polarity', 'Sentiment_Subjectivity']


# Understand the Text Data

For this NLP task:

```text
Translated_Review → Input text
Sentiment         → Target label
```

Before preprocessing, we should check missing values and the distribution of the sentiment labels.

In [38]:
df = reviews_df[["Translated_Review", "Sentiment"]].copy()
print("Missing values:")
display(df.isnull().sum())
print("\nSentiment distribution:")
display(df["Sentiment"].value_counts(dropna=False))

Missing values:


,0
Translated_Review,26868
Sentiment,26863



Sentiment distribution:


,count
Sentiment,
NaN,26863
Positive,23998
Negative,8271
Neutral,5163


In [39]:
df = df.dropna(subset=["Translated_Review", "Sentiment"]).copy()
df["Translated_Review"] = df["Translated_Review"].astype(str)
print("Shape after removing missing values:", df.shape)
display(df.head(10))

Shape after removing missing values: (37427, 2)


,Translated_Review,Sentiment
0,I like eat delicious food. That's I'm cooking ...,Positive
1,This help eating healthy exercise regular basis,Positive
3,Works great especially going grocery store,Positive
4,Best idea us,Positive
5,Best way,Positive
6,Amazing,Positive
8,"Looking forward app,",Neutral
9,It helpful site ! It help foods get !,Neutral
10,good you.,Positive
11,Useful information The amount spelling errors ...,Positive


#  Why Does Text Need Preprocessing?

A model cannot directly understand raw human language. Text must eventually be represented numerically.

Raw reviews may contain:

- Uppercase and lowercase variations.
- Punctuation.
- Extra spaces.
- Common words with limited information.
- Different grammatical forms of the same word.
- Words that are critical to sentiment, such as `not`.

For example:

"GOOD"
"Good"
"good!"


can represent the same idea, but a naive text-processing system may initially treat them as different forms.

The goal of preprocessing is therefore:

> **Create a more consistent representation of text while preserving information that matters for the task.**

For sentiment analysis, cleaning must be conservative because changing or removing a word can change the meaning of a review.

#  Word Tokenization

## What is Tokenization?

Tokenization splits a sentence into smaller units called **tokens**.

Example:

"The movie was great!"


becomes approximately:

["The", "movie", "was", "great", "!"]

Word tokenization is useful for traditional NLP preprocessing because it gives us individual words that can be filtered and normalized.

In [40]:
from nltk.tokenize import word_tokenize
sample_text = "The movie was great!"
tokens = word_tokenize(sample_text)
print("Original text:")
print(sample_text)
print("\nTokens:")
print(tokens)

Original text:
The movie was great!

Tokens:
['The', 'movie', 'was', 'great', '!']


## Why is Tokenization Important?

Tokenization creates the basic units that later operations work with.

The general flow is:

```text
Raw Text
   ↓
Tokenization
   ↓
Cleaning
   ↓
Normalization
   ↓
Numerical Representation
   ↓
Model
```

For traditional NLP pipelines, the tokens may later be converted into TF-IDF vectors, embeddings, or integer sequences.

For Transformers, tokenization is handled by the model's tokenizer.

# Sub-word Tokenization

Modern Transformer architectures like **DistilBERT** employ sub-word tokenization.

Rather than forcing each entire word to appear in the vocabulary, a word may be broken down into sub-pieces.

In theory:

```text
Word
 ↓
Sub-word pieces
 ↓
Token IDs
 ↓
Transformer
```

In doing so, the Transformer handles rare or unknown words better.

### Key difference

The word tokenizer illustrated above is good for teaching classical NLP tokenization.

But **DistilBERT has its own tokenizer**, and that tokenizer should be used whenever inputting text to DistilBERT.

In [41]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
sample_texts = ["This app is amazing!","I do not like this application."]
encoded = tokenizer(
    sample_texts,
    padding=True,
    truncation=True,
    return_tensors="pt")
print("Tokenizer output keys:")
print(encoded.keys())
print("\nInput IDs shape:")
print(encoded["input_ids"].shape)
print("\nFirst review tokens:")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]))

Tokenizer output keys:
KeysView({'input_ids': tensor([[  101,  2023, 10439,  2003,  6429,   999,   102,     0,     0],
        [  101,  1045,  2079,  2025,  2066,  2023,  4646,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])})

Input IDs shape:
torch.Size([2, 9])

First review tokens:
['[CLS]', 'this', 'app', 'is', 'amazing', '!', '[SEP]', '[PAD]', '[PAD]']


### Key Point Regarding DistilBERT

When dealing with a transformer model, one must not always use **aggressive pre-processing techniques**, such as removing all stop words or stemming/lemmatizing all words.

DistilBERT is pre-trained on natural language data and uses contextual information. Over-aggressiveness with pre-processing could impair the performance of the model.

Hence, this notebook covers all the necessary pre-processing concepts needed in NLP for Sprint 3, while also explaining why the overall pre-processing pipeline needs to be carefully designed.

#  Text Cleaning and Normalization

We will demonstrate the following steps:

1. Lowercasing.
2. Removing punctuation.
3. Handling unwanted characters.
4. Removing extra whitespace.

The goal is to make text more consistent.

### Example

```text
"THIS app is GREAT!!!"
```

can become:

```text
"this app is great"
```

In [42]:
sample_text = "THIS app is GREAT!!! 100% useful."
lowercased = sample_text.lower()
no_punctuation = lowercased.translate(str.maketrans("", "", string.punctuation))
normalized = re.sub(r"\s+", " ", no_punctuation).strip()
print("Original:")
print(sample_text)
print("\nLowercased:")
print(lowercased)
print("\nPunctuation removed:")
print(no_punctuation)
print("\nNormalized:")
print(normalized)

Original:
THIS app is GREAT!!! 100% useful.

Lowercased:
this app is great!!! 100% useful.

Punctuation removed:
this app is great 100 useful

Normalized:
this app is great 100 useful


## Are We Required to Remove All Numbers?

Not necessarily.

The removal of numbers is **contextual**.

For instance:

```text
"5 stars"
"10/10"
```

contains sentiment information.

Therefore, we will not blindly remove every number from the project reviews. The preprocessing pipeline will focus on normalization while preserving potentially useful information.

#  Stop Word Removal with Negation Words Preservation

The stop words are the very frequent words like:

```text
the
is
a
an
was
are
of
to
```

In some traditional NLP applications, their removal can decrease vocabulary and noise.

But for sentiment analysis, this step requires particular attention.

For example, consider the following sentences:

```text
"This app is good."
```

and

```text
"This app is not good."
```

If `not` is not preserved, both sentences can turn into

```text
"app good"
```

and their sentiment cannot be determined.

So, we will remove frequent stop words **without important negation words**.

In [43]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))
negation_words = {"not", "no", "never"}
stop_words_for_sentiment = stop_words - negation_words
print("Number of standard English stop words:", len(stop_words))
print("Number after protecting negations:", len(stop_words_for_sentiment))
print("\nProtected words:")
print(sorted(negation_words))
print("\nAre protected negations in the removal list?")
for word in sorted(negation_words):
    print(f"{word}: {word in stop_words_for_sentiment}")

Number of standard English stop words: 198
Number after protecting negations: 196

Protected words:
['never', 'no', 'not']

Are protected negations in the removal list?
never: False
no: False
not: False


In [44]:
sample = "The app is not good and I never recommend it."
tokens = word_tokenize(sample.lower())
filtered_tokens = [
    token
    for token in tokens
    if token.isalpha() and token not in stop_words_for_sentiment]
print("Original tokens:")
print(tokens)
print("\nAfter stop-word removal:")
print(filtered_tokens)

Original tokens:
['the', 'app', 'is', 'not', 'good', 'and', 'i', 'never', 'recommend', 'it', '.']

After stop-word removal:
['app', 'not', 'good', 'never', 'recommend']


### Interpretation

The result should still contain:

```text
not
never
```

This is important because these words can reverse or strongly modify sentiment.

This demonstrates an important NLP principle:

> **Preprocessing choices must depend on the task.**

For sentiment analysis, preserving negations is more important than blindly following a generic stop-word list.

# Lemmatization

## What is Lemmatization?

In lemmatization, words are reduced to their base or dictionary forms.

Examples:

```text
cars       → car
studies    → study
running    → run
```

Lemmatization is generally more meaningful than just removing word suffixes.

In [45]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
examples = [("cars", "n"),("studies", "n"),("running", "v"),("played", "v"),("better", "a")]
for word, pos in examples:
    print(f"{word:10} -> {lemmatizer.lemmatize(word, pos=pos)}")

cars       -> car
studies    -> study
running    -> run
played     -> play
better     -> good


## Why Does POS Matter?

POS means **Part of Speech**.

Examples:

- `n` → noun
- `v` → verb
- `a` → adjective
- `r` → adverb

The same word can sometimes have different grammatical roles. Providing the POS helps the lemmatizer choose a better base form.

For example:

```text
running → run
```

when `running` is treated as a verb.

#  Lemmatization and Stemming Differences

They both attempt to reduce various word forms.

### Stemming

The stemming process typically cuts off word endings through simple algorithms.

Can generate forms that are not actual English words.

Example:

```text
studies → studi
```

### Lemmatization

Attempts to obtain an actual dictionary form of a word.

Example:

```text
studies → study
```

### Comparison

| Type | Algorithm | Word Form Produced | Advantage | Disadvantage |
|---|---|---|---|---|
| Stemming | Removes word endings | studies → studi | Quick and easy | Can generate non-existent words |
| Lemmatization | Uses language knowledge | studies → study | Obtains meaningful words | Complicated process |

As far as this lesson goes, **lemmatization is preferred**.

In [46]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()
comparison_words = ["studies","running","played","cars"]
comparison = pd.DataFrame({"Word": comparison_words,"Stemmed": [stemmer.stem(w) for w in comparison_words],"Lemmatized": [lemmatizer.lemmatize(w, pos="v")for w in comparison_words]})
print(comparison)

      Word Stemmed Lemmatized
0  studies   studi      study
1  running     run        run
2   played    play       play
3     cars     car       cars


## Explicit Sub-word Tokenization Example

The following example uses a rare or longer word to show how the DistilBERT tokenizer can split a word into multiple sub-word pieces.


In [47]:
tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])


['[CLS]', 'this', 'app', 'is', 'amazing', '!', '[SEP]', '[PAD]', '[PAD]']

In [48]:
subword_examples = [
    "unbelievably",
    "customization",
    "unhappiness",
    "This application is unbelievably useful."
]

for text in subword_examples:
    print("Text:", text)
    print("Sub-word tokens:", tokenizer.tokenize(text))
    print("-" * 80)


Text: unbelievably
Sub-word tokens: ['un', '##bel', '##ie', '##va', '##bly']
--------------------------------------------------------------------------------
Text: customization
Sub-word tokens: ['custom', '##ization']
--------------------------------------------------------------------------------
Text: unhappiness
Sub-word tokens: ['un', '##ha', '##pp', '##iness']
--------------------------------------------------------------------------------
Text: This application is unbelievably useful.
Sub-word tokens: ['this', 'application', 'is', 'un', '##bel', '##ie', '##va', '##bly', 'useful', '.']
--------------------------------------------------------------------------------


### Interpretation

The tokenizer does not need every complete word to exist in its vocabulary. If a word is rare or unknown, DistilBERT can split it into smaller known pieces. This helps the model handle new, misspelled, or uncommon words more effectively.

This is different from the traditional word tokenizer, which usually treats a complete word as one token.


#  Complete NLP Pipeline for Preprocessing

Now, let’s put all of the above steps into one single function that can be reused.

The pipeline will:

1. Lowercase the text.
2. Tokenize the text.
3. Remove punctuation and non-alphanumeric tokens.
4. Remove stop words.
5. Preserve sentiment-related negations.
6. Lemmatize words.
7. Output the processed text.

### Important design choice

We will keep:

```text
not
no
never
```

since this is a sentiment-classification problem.

In [49]:
def preprocess_text(text):


    if pd.isna(text):
        return ""

    text = str(text).lower()

    tokens = word_tokenize(text)

    cleaned_tokens = []

    for token in tokens:

        if not token.isalpha():
            continue

        if token in stop_words_for_sentiment:
            continue

        lemma = lemmatizer.lemmatize(token, pos="n")

        lemma = lemmatizer.lemmatize(lemma, pos="v")

        cleaned_tokens.append(lemma)

    return " ".join(cleaned_tokens)


test_text = "The app is AMAZING!!! I do not like the new updates."

print("Original:")
print(test_text)

print("\nCleaned:")
print(preprocess_text(test_text))

Original:
The app is AMAZING!!! I do not like the new updates.

Cleaned:
app amaze not like new update


## Test the Pipeline with Several Examples

We should test the function on different types of reviews instead of only one sentence.

In [50]:
test_reviews = [
    "This app is AMAZING!!!",
    "I do not like this app.",
    "The application is very useful and easy to use.",
    "I never recommend this app.",
    "The update is not good at all.",
    "5 stars! Excellent application."
]

for review in test_reviews:
    print("Original :", review)
    print("Cleaned  :", preprocess_text(review))
    print("-" * 80)

Original : This app is AMAZING!!!
Cleaned  : app amaze
--------------------------------------------------------------------------------
Original : I do not like this app.
Cleaned  : not like app
--------------------------------------------------------------------------------
Original : The application is very useful and easy to use.
Cleaned  : application useful easy use
--------------------------------------------------------------------------------
Original : I never recommend this app.
Cleaned  : never recommend app
--------------------------------------------------------------------------------
Original : The update is not good at all.
Cleaned  : update not good
--------------------------------------------------------------------------------
Original : 5 stars! Excellent application.
Cleaned  : star excellent application
--------------------------------------------------------------------------------


#  Apply Preprocessing to the Project Dataset

We will now create a new column:

clean_text


The original `Translated_Review` column will remain unchanged.

This is important because we want to be able to compare the original text with the processed text.

In [51]:
df["clean_text"] = df["Translated_Review"].apply(preprocess_text)
display(df[["Translated_Review", "clean_text", "Sentiment"]].head(10))

,Translated_Review,clean_text,Sentiment
0,I like eat delicious food. That's I'm cooking ...,like eat delicious food cook food case best fo...,Positive
1,This help eating healthy exercise regular basis,help eat healthy exercise regular basis,Positive
3,Works great especially going grocery store,work great especially go grocery store,Positive
4,Best idea us,best idea u,Positive
5,Best way,best way,Positive
6,Amazing,amaze,Positive
8,"Looking forward app,",look forward app,Neutral
9,It helpful site ! It help foods get !,helpful site help food get,Neutral
10,good you.,good,Positive
11,Useful information The amount spelling errors ...,useful information amount spell error question...,Positive


#  Pre-Processing Comparison

The most effective verification process would be to compare the original review with the one that has been cleaned.

What we need to find is:

- Text in lowercase.
- Punctuation removed.
- Low-information stop words removed.
- Word variations minimized.
- Sentiment-relevant words kept.

In [52]:
comparison_df = df[["Translated_Review", "clean_text", "Sentiment"]].sample(min(15, len(df)),random_state=42)
display(comparison_df)

,Translated_Review,clean_text,Sentiment
31799,Great game heats phone short time. Please rect...,great game heat phone short time please rectif...,Positive
5707,This maths formulas I want,math formula want,Neutral
62076,Some suggestions improvement 1. Change throttl...,suggestion improvement change throttle adjust ...,Negative
18833,"The notifications work cellphone... Otherwise,...",notification work cellphone otherwise like,Neutral
11216,This helps speak Polish friends,help speak polish friend,Neutral
41525,Thanks continuing provide quality support. Sti...,thank continue provide quality support still f...,Positive
18131,Love game,love game,Positive
60819,It's good game much need update. There ads die...,good game much need update ad die get next lev...,Positive
14486,Make a spirit,make spirit,Neutral
36078,"I love it, dislike ingame purchases. I wish co...",love dislike ingame purchase wish could spend ...,Positive


## Vocabulary Size Before and After

One purpose of normalization is to reduce unnecessary variation in the vocabulary.
We can compare the number of unique tokens before and after preprocessing.

In [53]:
def tokenize_words(text):
    return [
        token
        for token in word_tokenize(str(text).lower())
        if token.isalpha()
    ]

original_tokens = []

for text in df["Translated_Review"].sample(
    min(5000, len(df)),
    random_state=42
):
    original_tokens.extend(tokenize_words(text))

clean_tokens = []

for text in df["clean_text"].sample(
    min(5000, len(df)),
    random_state=42
):
    clean_tokens.extend(text.split())

original_vocab = set(original_tokens)
clean_vocab = set(clean_tokens)

print("Sampled original token count:", len(original_tokens))
print("Sampled cleaned token count:", len(clean_tokens))

print("\nOriginal vocabulary size:", len(original_vocab))
print("Cleaned vocabulary size:", len(clean_vocab))

Sampled original token count: 90147
Sampled cleaned token count: 74651

Original vocabulary size: 8454
Cleaned vocabulary size: 6240


### Interpretation

A smaller vocabulary is not automatically better.

The goal is to remove unnecessary variation **without removing useful information**.

For sentiment classification, preserving words that influence meaning is more important than achieving the smallest possible vocabulary.

# Verify Task-Critical Words

This is an important requirement for today's lab.
We will explicitly verify that negation words such as:

```text
not
no
never
```

are preserved.

These words can change sentiment meaning.

In [54]:
negation_test_sentences = [
    "I do not like this app.",
    "No, this application is not useful.",
    "I never recommend this app."
]

for sentence in negation_test_sentences:
    cleaned = preprocess_text(sentence)

    print("Original :", sentence)
    print("Cleaned  :", cleaned)

    for word in ["not", "no", "never"]:
        if word in sentence.lower():
            print(f"  '{word}' preserved:", word in cleaned.split())



Original : I do not like this app.
Cleaned  : not like app
  'not' preserved: True
  'no' preserved: False
Original : No, this application is not useful.
Cleaned  : no application not useful
  'not' preserved: True
  'no' preserved: True
Original : I never recommend this app.
Cleaned  : never recommend app
  'never' preserved: True


In [55]:
assert "not" in preprocess_text("I do not like this app.").split()
assert "no" in preprocess_text("No, I do not recommend this app.").split()
assert "never" in preprocess_text("I never recommend this app.").split()
print("All negation-preservation tests passed successfully.")

All negation-preservation tests passed successfully.


## Why This Verification Matters

Suppose:

```text
"I like this app."
```

becomes:

```text
"like app"
```

That may still preserve the general sentiment.

But:

```text
"I do not like this app."
```

must not become:

```text
"like app"
```

because the word `not` changes the meaning.

Therefore, preprocessing must be evaluated based on the actual NLP task rather than applied blindly.

#  Document Preprocessing Decisions

## Preprocessing Decisions

| Step | Decision | Reason |
|---|---|---|
| Lowercasing | Applied | Reduces case-based variation |
| Tokenization | Applied | Converts text into manageable tokens |
| Punctuation removal | Applied for the traditional preprocessing demonstration | Reduces non-word noise |
| Number removal | Applied in the current traditional preprocessing pipeline | The implementation keeps alphabetic tokens only; however, numbers may be preserved in future task-specific versions if they contain useful sentiment information |
| Stop-word removal | Applied selectively | Reduces common words while preserving sentiment information |
| Negations | `not`, `no`, `never` preserved | They can change sentiment meaning |
| Lemmatization | Applied | Reduces word-form variation while keeping meaningful base forms |
| Stemming | Not used in the final pipeline | It can produce incomplete or non-dictionary forms |


---

## Important Transformer Consideration

The preprocessing pipeline above demonstrates the traditional NLP techniques required for this lesson.

However, the project's selected core model is **DistilBERT**.

DistilBERT uses its own sub-word tokenizer and relies on contextual information. Therefore, the final Transformer pipeline should avoid unnecessary aggressive cleaning.

For example, instead of manually removing every stop word before DistilBERT, we normally preserve the review text and let the Transformer tokenizer create the appropriate token representation.

### Conceptual final pipeline

```text
Google Play Review
        ↓
Minimal / Task-aware text normalization
        ↓
DistilBERT Tokenizer
        ↓
Input IDs + Attention Mask
        ↓
DistilBERT
        ↓
Classification Head
        ↓
Negative / Neutral / Positive
        ↓
Evaluation
```

This distinction is important because the goal of preprocessing is not simply to remove as much text as possible. The goal is to prepare useful information for the chosen model.

#  Day 1 Requirements Checklist

## Learning Objectives

- [x] Complete Sprint 3 planning.
- [x] Define the Sprint 3 goal.
- [x] Define the Sprint 3 backlog.
- [x] Carry forward improvements from Sprint 2.
- [x] Explain why text needs preprocessing.
- [x] Apply word tokenization.
- [x] Explain sub-word tokenization.
- [x] Apply lowercasing.
- [x] Remove punctuation in the traditional preprocessing pipeline.
- [x] Handle stop words.
- [x] Preserve sentiment-critical negations.
- [x] Apply lemmatization.
- [x] Compare lemmatization and stemming.
- [x] Build a complete preprocessing function.
- [x] Apply preprocessing to the Google Play review dataset.
- [x] Compare original and cleaned text.
- [x] Verify that task-critical words are preserved.
- [x] Document preprocessing decisions.



### Task Completion Summary

- **Sprint 3 Planning:** The sprint goal and backlog were defined, and the main improvements from Sprint 2 were carried forward.
- **Tokenization:** Word tokenization was applied to the project reviews using NLTK, and sub-word tokenization was demonstrated using the DistilBERT tokenizer.
- **Text Cleaning:** A full preprocessing function was created to lowercase text, tokenize it, remove punctuation and stop words, and apply lemmatization.
- **Dataset Processing:** The preprocessing function was applied to the `Translated_Review` column, and the results were stored in the new `clean_text` column.
- **Negation Verification:** The words `not`, `no`, and `never` were preserved and tested because they are important for sentiment classification.
- **Documentation:** The preprocessing decisions, limitations, and connection to DistilBERT were documented in Markdown.


# Final Summary

Sprint 3 introduced a complete NLP preprocessing workflow for the Google Play reviews.

The notebook covered tokenization, lowercasing, punctuation and stop-word handling, lemmatization, and preservation of sentiment-critical negations such as `not`, `no`, and `never`.

It also demonstrated that preprocessing should be task-dependent. Traditional preprocessing is useful for learning and classical NLP, while DistilBERT relies on its own sub-word tokenizer and requires less aggressive text cleaning.

---

## Day 1 Outcome

At the end of this notebook, we have:

```text
Sprint 3 Plan
     ↓
Google Play Reviews
     ↓
Text Inspection
     ↓
Tokenization
     ↓
Cleaning & Normalization
     ↓
Stop-word Handling
     ↓
Negation Preservation
     ↓
Lemmatization
     ↓
Before / After Verification
     ↓
Documented Preprocessing Decisions
```

**Day 1 completed successfully.**